In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

from abstract_values.utils.data import BIDS_FOLDER

In [ ]:
# ── settings ───────────────────────────────────────────────────────────────
SUBJECTS  = ['pil01', 'pil02']
N_VOXELS  = 100
NOISE     = 'full'   # 'full' or 'spherical'
SMOOTHED  = False
LAMBD     = 0.1      # default λ for noise-model regularisation

DERIV_DIR = BIDS_FOLDER / 'derivatives' / 'decoding'

ROIS = ['BensonV1', 'BensonV2', 'BensonV3', 'NPCl', 'NPCr']
ROI_COLORS = {
    'BensonV1': '#1f77b4',
    'BensonV2': '#ff7f0e',
    'BensonV3': '#2ca02c',
    'NPCl':     '#d62728',
    'NPCr':     '#9467bd',
}
MAPPING_COLORS = {'cdf': 'steelblue', 'inv_cdf': 'tomato', 'inverse_cdf': 'tomato'}


def get_mapping(subject, session):
    """Return 'cdf' or 'inverse_cdf' for this subject + session.

    Even subject number → ses-1 = cdf,         ses-2 = inverse_cdf
    Odd  subject number → ses-1 = inverse_cdf, ses-2 = cdf
    """
    num = int(''.join(c for c in str(subject) if c.isdigit()))
    if num % 2 == 0:
        return 'cdf' if session == 1 else 'inverse_cdf'
    return 'inverse_cdf' if session == 1 else 'cdf'

In [ ]:
def load_decoding(kind, subject, mask, n_voxels=N_VOXELS, noise=NOISE,
                  smoothed=SMOOTHED, lambd=LAMBD):
    """Load all available decoding TSVs for this subject/mask combo.

    Searches ses-* subdirectories and the flat sub/func path.
    Concatenates and deduplicates by index.

    Index : (session, run, trial_nr, true_x)
    Columns : float (orientation rad or CHF value)
    """
    smooth_label = '_smoothed' if smoothed else ''
    lambd_label  = f'_lambda-{lambd}' if lambd != 0.0 else ''
    sub_dir = DERIV_DIR / kind / f'sub-{subject}'
    if not sub_dir.exists():
        return None

    def _load(fn):
        df = pd.read_csv(fn, sep='\t', index_col=list(range(4)))
        df.columns = df.columns.astype(float)
        return df

    dfs = []
    for sd in sorted(sub_dir.iterdir()):
        if not sd.is_dir():
            continue
        fn = (sd / 'func' /
              f'sub-{subject}_{sd.name}_mask-{mask}'
              f'_nvoxels-{n_voxels}_noise-{noise}{smooth_label}{lambd_label}_pars.tsv')
        if fn.exists():
            dfs.append(_load(fn))
    fn_all = (sub_dir / 'func' /
              f'sub-{subject}_mask-{mask}'
              f'_nvoxels-{n_voxels}_noise-{noise}{smooth_label}{lambd_label}_pars.tsv')
    if fn_all.exists():
        dfs.append(_load(fn_all))
    if not dfs:
        return None
    combined = pd.concat(dfs).sort_index()
    return combined[~combined.index.duplicated(keep='first')]


def normalize_rows(df):
    return df.div(df.sum(axis=1), axis=0)


def posterior_mean_orientation(df):
    """Circular mean of posterior for axial orientations (period=π, columns in rad)."""
    cols = df.columns.values.astype(float)
    norm = normalize_rows(df).values
    sin_w = norm @ np.sin(2 * cols)
    cos_w = norm @ np.cos(2 * cols)
    return np.arctan2(sin_w, cos_w) / 2 % np.pi


def posterior_mean_value(df):
    """Expected value (linear mean) of posterior (columns in CHF)."""
    cols = df.columns.values.astype(float)
    return normalize_rows(df).values @ cols


def circular_error_deg(true_rad, pred_rad):
    """Mean absolute circular error in degrees for axial orientations (period=π)."""
    diff = np.abs(true_rad - pred_rad) % np.pi
    diff = np.where(diff > np.pi / 2, np.pi - diff, diff)
    return float(np.rad2deg(np.mean(diff)))


def circular_correlation(true_rad, pred_rad):
    """Circular-circular correlation for axial orientations (period=π).

    Doubles angles to map period-π to full circle, then averages
    cos- and sin-component Pearson r.
    """
    t2, p2 = 2 * true_rad, 2 * pred_rad
    r_cos = np.corrcoef(np.cos(t2), np.cos(p2))[0, 1]
    r_sin = np.corrcoef(np.sin(t2), np.sin(p2))[0, 1]
    return float((r_cos + r_sin) / 2)


def aggregate_posterior(df):
    """Align each trial posterior to its true stimulus and return mean.

    Returns: (deviation_axis, mean_posterior)
    """
    norm = normalize_rows(df)
    cols = norm.columns.values.astype(float)
    true_vals = df.index.get_level_values(-1).astype(float)
    n = len(cols)
    aligned = np.zeros((len(norm), n))
    for i, (row, tv) in enumerate(zip(norm.values, true_vals)):
        ci = int(np.argmin(np.abs(cols - tv)))
        aligned[i] = np.roll(row, n // 2 - ci)
    step = cols[1] - cols[0]
    deviation = (np.arange(n) - n // 2) * step
    return deviation, aligned.mean(axis=0)

# Gabor orientation decoding

In [ ]:
# ── aggregate posterior per ROI × subject ──────────────────────────────────
fig, axes = plt.subplots(len(SUBJECTS), len(ROIS),
                          figsize=(3.5 * len(ROIS), 3.5 * len(SUBJECTS)),
                          constrained_layout=True, sharey='row')

for row_i, subject in enumerate(SUBJECTS):
    for col_i, roi in enumerate(ROIS):
        ax = axes[row_i][col_i]
        df = load_decoding('gabor', subject, roi)
        if df is None:
            ax.text(0.5, 0.5, 'no data', transform=ax.transAxes,
                    ha='center', va='center', color='gray')
            if row_i == 0:
                ax.set_title(roi, fontsize=10)
            continue

        for ses in sorted(df.index.get_level_values('session').unique()):
            ses_df  = df.xs(ses, level='session')
            mapping = get_mapping(subject, ses)
            dev, post = aggregate_posterior(ses_df)
            ax.plot(np.rad2deg(dev), post, lw=2,
                    color=MAPPING_COLORS[mapping],
                    label=f'ses-{ses} ({mapping})')

        ax.axhline(1 / df.shape[1], color='k', lw=0.8, ls='--', alpha=0.4)
        ax.axvline(0, color='k', lw=0.8, ls=':', alpha=0.3)
        if row_i == len(SUBJECTS) - 1:
            ax.set_xlabel('Orientation error (°)', fontsize=9)
        if col_i == 0:
            ax.set_ylabel(f'sub-{subject}\nMean posterior', fontsize=9)
        if row_i == 0:
            ax.set_title(roi, fontsize=10)
        if row_i == 0 and col_i == 0:
            ax.legend(fontsize=7)

fig.suptitle(f'Gabor decoding: aggregate posterior  '
             f'[n={N_VOXELS}, noise={NOISE}, λ={LAMBD}]', fontsize=12)
plt.show()

In [ ]:
# ── scatter: decoded orientation vs. true orientation ──────────────────────
pairs = sorted({(s, ses)
                for s in SUBJECTS
                for roi in ROIS
                for df in [load_decoding('gabor', s, roi)]
                if df is not None
                for ses in df.index.get_level_values('session').unique()})

ncols = max(len(pairs), 1)
fig, axes = plt.subplots(1, ncols, figsize=(4.5 * ncols, 4.5),
                          constrained_layout=True)
if ncols == 1:
    axes = [axes]

for ax, (subject, ses) in zip(axes, pairs):
    mapping = get_mapping(subject, ses)
    for roi in ROIS:
        df = load_decoding('gabor', subject, roi)
        if df is None or ses not in df.index.get_level_values('session'):
            continue
        ses_df   = df.xs(ses, level='session')
        true_deg = np.rad2deg(ses_df.index.get_level_values(
            'true_orientation_rad').astype(float))
        pred_deg = np.rad2deg(posterior_mean_orientation(ses_df))
        ax.scatter(true_deg, pred_deg, s=5, alpha=0.4,
                   color=ROI_COLORS.get(roi, 'gray'), label=roi)

    ax.plot([0, 180], [0, 180], 'k--', lw=1, alpha=0.5)
    ax.set_xlim(0, 180)
    ax.set_ylim(0, 180)
    ax.set_aspect('equal')
    ax.set_xlabel('True orientation (°)', fontsize=10)
    ax.set_ylabel('Decoded orientation (°)', fontsize=10)
    ax.set_title(f'sub-{subject}  ses-{ses} ({mapping})', fontsize=10)
    ax.legend(fontsize=7, markerscale=3, loc='upper left')

fig.suptitle(f'Gabor decoding scatter (posterior mean)  '
             f'[n={N_VOXELS}, noise={NOISE}, λ={LAMBD}]', fontsize=12)
plt.show()

In [ ]:
# ── accuracy table: circular error + circular correlation per ROI × session
records = []
for subject in SUBJECTS:
    for roi in ROIS:
        df = load_decoding('gabor', subject, roi)
        if df is None:
            continue
        for ses in sorted(df.index.get_level_values('session').unique()):
            ses_df   = df.xs(ses, level='session')
            true_rad = ses_df.index.get_level_values(
                'true_orientation_rad').astype(float)
            pred_rad = posterior_mean_orientation(ses_df)
            records.append(dict(
                subject      = subject,
                session      = ses,
                mapping      = get_mapping(subject, ses),
                roi          = roi,
                n_trials     = len(ses_df),
                circ_err_deg = circular_error_deg(true_rad, pred_rad),
                circ_r       = circular_correlation(true_rad, pred_rad),
            ))

summary_gabor = pd.DataFrame(records)
print(summary_gabor.to_string(index=False))

In [ ]:
# ── bar: circular error per ROI × subject × session ────────────────────────
CHANCE_DEG = 45.0   # uniform over 25 orientations spaced 7.5°

fig, axes = plt.subplots(1, len(SUBJECTS), figsize=(5.5 * len(SUBJECTS), 4),
                          constrained_layout=True, sharey=True)
if len(SUBJECTS) == 1:
    axes = [axes]

for ax, subject in zip(axes, SUBJECTS):
    sub_df   = summary_gabor[summary_gabor['subject'] == subject]
    x        = np.arange(len(ROIS))
    sessions = sorted(sub_df['session'].unique())
    width    = 0.8 / max(len(sessions), 1)

    for i, ses in enumerate(sessions):
        ses_data = sub_df[sub_df['session'] == ses].set_index('roi')
        mapping  = ses_data['mapping'].iloc[0] if len(ses_data) else 'cdf'
        errs = [ses_data.loc[roi, 'circ_err_deg']
                if roi in ses_data.index else np.nan for roi in ROIS]
        offset = (i - (len(sessions) - 1) / 2) * width
        ax.bar(x + offset, errs, width, label=f'ses-{ses} ({mapping})',
               color=MAPPING_COLORS[mapping], alpha=0.85)

    ax.axhline(CHANCE_DEG, color='k', lw=1.5, ls='--', alpha=0.5,
               label=f'chance (~{CHANCE_DEG:.0f}°)')
    ax.set_xticks(x)
    ax.set_xticklabels(ROIS, rotation=30, ha='right', fontsize=9)
    ax.set_ylabel('Mean circular error (°)', fontsize=10)
    ax.set_title(f'sub-{subject}', fontsize=11)
    ax.legend(fontsize=8)
    ax.set_ylim(0, CHANCE_DEG * 1.1)

fig.suptitle(f'Gabor decoding error  [n={N_VOXELS}, noise={NOISE}, λ={LAMBD}]',
             fontsize=12)
plt.show()

# Abstract value decoding

In [ ]:
# ── aggregate posterior per ROI × subject ──────────────────────────────────
fig, axes = plt.subplots(len(SUBJECTS), len(ROIS),
                          figsize=(3.5 * len(ROIS), 3.5 * len(SUBJECTS)),
                          constrained_layout=True, sharey='row')

for row_i, subject in enumerate(SUBJECTS):
    for col_i, roi in enumerate(ROIS):
        ax = axes[row_i][col_i]
        df = load_decoding('value', subject, roi)
        if df is None:
            ax.text(0.5, 0.5, 'no data', transform=ax.transAxes,
                    ha='center', va='center', color='gray')
            if row_i == 0:
                ax.set_title(roi, fontsize=10)
            continue

        for ses in sorted(df.index.get_level_values('session').unique()):
            ses_df  = df.xs(ses, level='session')
            mapping = get_mapping(subject, ses)
            dev, post = aggregate_posterior(ses_df)
            ax.plot(dev, post, lw=2,
                    color=MAPPING_COLORS[mapping],
                    label=f'ses-{ses} ({mapping})')

        ax.axhline(1 / df.shape[1], color='k', lw=0.8, ls='--', alpha=0.4)
        ax.axvline(0, color='k', lw=0.8, ls=':', alpha=0.3)
        if row_i == len(SUBJECTS) - 1:
            ax.set_xlabel('Value error (CHF)', fontsize=9)
        if col_i == 0:
            ax.set_ylabel(f'sub-{subject}\nMean posterior', fontsize=9)
        if row_i == 0:
            ax.set_title(roi, fontsize=10)
        if row_i == 0 and col_i == 0:
            ax.legend(fontsize=7)

fig.suptitle(f'Value decoding: aggregate posterior  '
             f'[n={N_VOXELS}, noise={NOISE}, λ={LAMBD}]', fontsize=12)
plt.show()

In [ ]:
# ── scatter: decoded value vs. true value ──────────────────────────────────
pairs_val = sorted({(s, ses)
                    for s in SUBJECTS
                    for roi in ROIS
                    for df in [load_decoding('value', s, roi)]
                    if df is not None
                    for ses in df.index.get_level_values('session').unique()})

ncols = max(len(pairs_val), 1)
fig, axes = plt.subplots(1, ncols, figsize=(4.5 * ncols, 4.5),
                          constrained_layout=True)
if ncols == 1:
    axes = [axes]

for ax, (subject, ses) in zip(axes, pairs_val):
    mapping = get_mapping(subject, ses)
    for roi in ROIS:
        df = load_decoding('value', subject, roi)
        if df is None or ses not in df.index.get_level_values('session'):
            continue
        ses_df = df.xs(ses, level='session')
        true_v = ses_df.index.get_level_values('true_value_chf').astype(float)
        pred_v = posterior_mean_value(ses_df)
        ax.scatter(true_v, pred_v, s=5, alpha=0.4,
                   color=ROI_COLORS.get(roi, 'gray'), label=roi)

    vmin, vmax = 2.0, 42.0
    ax.plot([vmin, vmax], [vmin, vmax], 'k--', lw=1, alpha=0.5)
    ax.set_xlim(vmin, vmax)
    ax.set_ylim(vmin, vmax)
    ax.set_aspect('equal')
    ax.set_xlabel('True value (CHF)', fontsize=10)
    ax.set_ylabel('Decoded value (CHF)', fontsize=10)
    ax.set_title(f'sub-{subject}  ses-{ses} ({mapping})', fontsize=10)
    ax.legend(fontsize=7, markerscale=3, loc='upper left')

fig.suptitle(f'Value decoding scatter (posterior mean)  '
             f'[n={N_VOXELS}, noise={NOISE}, λ={LAMBD}]', fontsize=12)
plt.show()

In [ ]:
# ── accuracy table: MAE + r per ROI × session ──────────────────────────────
records_val = []
for subject in SUBJECTS:
    for roi in ROIS:
        df = load_decoding('value', subject, roi)
        if df is None:
            continue
        for ses in sorted(df.index.get_level_values('session').unique()):
            ses_df  = df.xs(ses, level='session')
            true_v  = ses_df.index.get_level_values('true_value_chf').astype(float)
            pred_v  = posterior_mean_value(ses_df)
            r, _    = pearsonr(true_v, pred_v)
            records_val.append(dict(
                subject  = subject,
                session  = ses,
                mapping  = get_mapping(subject, ses),
                roi      = roi,
                n_trials = len(ses_df),
                mae_chf  = float(np.mean(np.abs(true_v - pred_v))),
                r        = r,
            ))

summary_value = pd.DataFrame(records_val)
print(summary_value.to_string(index=False))

In [ ]:
# ── bar: MAE per ROI × subject × session ───────────────────────────────────
CHANCE_CHF = 10.0   # MAE of uniform posterior over ~40 CHF range

fig, axes = plt.subplots(1, len(SUBJECTS), figsize=(5.5 * len(SUBJECTS), 4),
                          constrained_layout=True, sharey=True)
if len(SUBJECTS) == 1:
    axes = [axes]

for ax, subject in zip(axes, SUBJECTS):
    sub_df   = summary_value[summary_value['subject'] == subject]
    x        = np.arange(len(ROIS))
    sessions = sorted(sub_df['session'].unique())
    width    = 0.8 / max(len(sessions), 1)

    for i, ses in enumerate(sessions):
        ses_data = sub_df[sub_df['session'] == ses].set_index('roi')
        mapping  = ses_data['mapping'].iloc[0] if len(ses_data) else 'cdf'
        maes = [ses_data.loc[roi, 'mae_chf']
                if roi in ses_data.index else np.nan for roi in ROIS]
        offset = (i - (len(sessions) - 1) / 2) * width
        ax.bar(x + offset, maes, width, label=f'ses-{ses} ({mapping})',
               color=MAPPING_COLORS[mapping], alpha=0.85)

    ax.axhline(CHANCE_CHF, color='k', lw=1.5, ls='--', alpha=0.5,
               label=f'chance (~{CHANCE_CHF:.0f} CHF)')
    ax.set_xticks(x)
    ax.set_xticklabels(ROIS, rotation=30, ha='right', fontsize=9)
    ax.set_ylabel('MAE (CHF)', fontsize=10)
    ax.set_title(f'sub-{subject}', fontsize=11)
    ax.legend(fontsize=8)
    ax.set_ylim(0, CHANCE_CHF * 1.5)

fig.suptitle(f'Value decoding error  [n={N_VOXELS}, noise={NOISE}, λ={LAMBD}]',
             fontsize=12)
plt.show()

# Effect of λ (noise regularisation): λ=0.0 vs λ=0.1

In [ ]:
# Compare λ=0 and λ=0.1 on aggregate posteriors (BensonV1 + NPCr).
LAMBDA_GRID  = [0.0, 0.1]
SWEEP_ROIS_L = ['BensonV1', 'NPCr']

n_panels = len(SUBJECTS) * len(SWEEP_ROIS_L)
fig, axes = plt.subplots(2, n_panels,
                          figsize=(4.5 * n_panels, 8),
                          constrained_layout=True)

col_map = {(s, r): i for i, (s, r) in
           enumerate((s, r) for s in SUBJECTS for r in SWEEP_ROIS_L)}

for kind_i, kind in enumerate(['gabor', 'value']):
    for subject in SUBJECTS:
        for roi in SWEEP_ROIS_L:
            ax = axes[kind_i][col_map[(subject, roi)]]
            for lam in LAMBDA_GRID:
                df = load_decoding(kind, subject, roi, lambd=lam)
                if df is None:
                    continue
                for ses in sorted(df.index.get_level_values('session').unique()):
                    ses_df  = df.xs(ses, level='session')
                    mapping = get_mapping(subject, ses)
                    dev, post = aggregate_posterior(ses_df)
                    x_axis = np.rad2deg(dev) if kind == 'gabor' else dev
                    ax.plot(x_axis, post, lw=2,
                            ls='-' if lam == 0.1 else '--',
                            color=MAPPING_COLORS[mapping],
                            label=f'λ={lam} ses-{ses} ({mapping})')

            ax.axvline(0, color='k', lw=0.8, ls=':', alpha=0.3)
            ax.set_xlabel('Error (°)' if kind == 'gabor' else 'Error (CHF)',
                          fontsize=9)
            ax.set_title(f'{kind} | sub-{subject} | {roi}', fontsize=9)
            ax.legend(fontsize=6, ncol=2)

fig.suptitle('λ comparison: aggregate posteriors  (solid=0.1, dashed=0.0)',
             fontsize=12)
plt.show()

In [ ]:
# ── λ comparison: accuracy summary table ──────────────────────────────────
lam_records = []
for kind in ['gabor', 'value']:
    for subject in SUBJECTS:
        for roi in SWEEP_ROIS_L:
            for lam in LAMBDA_GRID:
                df = load_decoding(kind, subject, roi, lambd=lam)
                if df is None:
                    continue
                for ses in sorted(df.index.get_level_values('session').unique()):
                    ses_df  = df.xs(ses, level='session')
                    mapping = get_mapping(subject, ses)
                    row = dict(kind=kind, subject=subject, roi=roi,
                               session=ses, mapping=mapping, lambd=lam)
                    if kind == 'gabor':
                        true_rad = ses_df.index.get_level_values(
                            'true_orientation_rad').astype(float)
                        pred_rad = posterior_mean_orientation(ses_df)
                        row['err'] = circular_error_deg(true_rad, pred_rad)
                        row['r']   = circular_correlation(true_rad, pred_rad)
                    else:
                        true_v = ses_df.index.get_level_values(
                            'true_value_chf').astype(float)
                        pred_v = posterior_mean_value(ses_df)
                        row['err']    = float(np.mean(np.abs(true_v - pred_v)))
                        row['r'], _   = pearsonr(true_v, pred_v)
                    lam_records.append(row)

lam_df = pd.DataFrame(lam_records)
print(lam_df.to_string(index=False))

# Effect of n_voxels on decoding accuracy

In [ ]:
N_VOXELS_GRID = [50, 100, 250, 500]
SWEEP_ROIS_N  = ['BensonV1', 'NPCr']

fig, axes = plt.subplots(2, len(SUBJECTS),
                          figsize=(5 * len(SUBJECTS), 8),
                          constrained_layout=True)
units = {'gabor': '°', 'value': 'CHF'}

for kind_i, kind in enumerate(['gabor', 'value']):
    for sub_i, subject in enumerate(SUBJECTS):
        ax = axes[kind_i][sub_i]
        sub = Subject(subject, bids_folder=BIDS_FOLDER)

        for roi in SWEEP_ROIS_N:
            errs, nvox_found = [], []
            for nv in N_VOXELS_GRID:
                df = load_decoding(kind, subject, roi, n_voxels=nv)
                if df is None:
                    continue
                if kind == 'gabor':
                    true_v = df.index.get_level_values(
                        'true_orientation_rad').astype(float)
                    pred_v = posterior_mean_orientation(df)
                    err = circular_error_deg(true_v, pred_v)
                else:
                    true_v = df.index.get_level_values(
                        'true_value_chf').astype(float)
                    pred_v = posterior_mean_value(df)
                    err = float(np.mean(np.abs(true_v - pred_v)))
                errs.append(err)
                nvox_found.append(nv)

            if errs:
                ax.plot(nvox_found, errs, 'o-', lw=2,
                        color=ROI_COLORS.get(roi, 'gray'), label=roi)

        ax.set_xlabel('n_voxels', fontsize=10)
        ax.set_ylabel(f'MAE ({units[kind]})', fontsize=10)
        ax.set_title(f'{kind} decoding  sub-{subject}', fontsize=10)
        ax.set_xscale('log')
        ax.set_xticks(N_VOXELS_GRID)
        ax.set_xticklabels(N_VOXELS_GRID)
        ax.legend(fontsize=9)

fig.suptitle(f'Decoding error vs. n_voxels  [noise={NOISE}, λ={LAMBD}]',
             fontsize=12)
plt.show()